In [28]:
from datasets import load_dataset
from pathlib import Path
from tqdm.auto import tqdm
import re
import unicodedata
import random


## Here is set up the tiny stories training, test and validation data. I used 10 million stories to generate 15 million sentences. The N-gram model learned 33 thousand unique words. 

In [29]:


def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)

    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')

    return text


def split_story_into_sentences(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")

    # Split before removing punctuation.
    sentences = re.split(r"[.!?]+", text)

    return sentences


def clean_sentence(sentence):
    sentence = normalize_text(sentence)
    sentence = sentence.lower()

    # Remove apostrophes after sentence splitting.
    sentence = sentence.replace("'", "")

    # Keep only letters, numbers, and spaces.
    sentence = re.sub(r"[^a-z0-9\s]", " ", sentence)

    # Normalize whitespace.
    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence


def story_to_clean_sentences(story_text):
    raw_sentences = split_story_into_sentences(story_text)

    cleaned_sentences = []

    for sentence in raw_sentences:
        cleaned = clean_sentence(sentence)

        if cleaned:
            cleaned_sentences.append(cleaned)

    return cleaned_sentences


def save_lines(lines, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")


def convert_stories_to_sentences(stories):
    all_sentences = []

    for story in stories:
        sentences = story_to_clean_sentences(story)
        all_sentences.extend(sentences)

    return all_sentences


def main():
    output_dir = Path("data")
    output_dir.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset("roneneldan/TinyStories", split="train[:1000000]")

    stories = []

    for row in tqdm(dataset, desc="Loading TinyStories"):
        text = row["text"].strip()

        if text:
            stories.append(text)

    random.seed(42)
    random.shuffle(stories)

    n = len(stories)

    train_end = int(0.8 * n)
    val_end = int(0.81 * n)

    train_stories = stories[:train_end]
    val_stories = stories[train_end:val_end]
    test_stories = stories[val_end:]

    train_sentences = convert_stories_to_sentences(train_stories)
    val_sentences = convert_stories_to_sentences(val_stories)
    test_sentences = convert_stories_to_sentences(test_stories)

    # Optional if you want exactly 2000 test sentences

    save_lines(train_sentences, "data/tiny_stories/tinystories_train.txt")
    save_lines(val_sentences, "data/tiny_stories/tinystories_val.txt")
    save_lines(test_sentences, "data/tiny_stories/tinystories_test.txt")
    print("Saved splits:")
    print("Train stories:", len(train_stories))
    print("Val stories:  ", len(val_stories))
    print("Test stories: ", len(test_stories))
    print()
    print("Train sentences:", len(train_sentences))
    print("Val sentences:  ", len(val_sentences))
    print("Test sentences: ", len(test_sentences))


if __name__ == "__main__":
    main()

Loading TinyStories:   0%|          | 0/1000000 [00:00<?, ?it/s]

Saved splits:
Train stories: 799943
Val stories:   9999
Test stories:  189987

Train sentences: 15603516
Val sentences:   193167
Test sentences:  3706169


## Same thing with Wikitext-2, this is for N-gram

In [27]:



def split_text_into_sentences(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")

    # Split before removing punctuation.
    sentences = re.split(r"[.!?]+", text)

    return sentences




def text_to_clean_sentences(text):
    raw_sentences = split_text_into_sentences(text)

    cleaned_sentences = []

    for sentence in raw_sentences:
        cleaned = clean_sentence(sentence)

        if cleaned:
            cleaned_sentences.append(cleaned)

    return cleaned_sentences




def convert_wikitext_split_to_sentences(dataset_split, split_name):
    all_sentences = []

    for row in tqdm(dataset_split, desc=f"Processing WikiText-2 {split_name}"):
        text = row["text"].strip()

        if not text:
            continue

        # WikiText has article headings like:
        # = Valkyria Chronicles III =
        # = = Gameplay = =
        # We remove those because they are not normal sentences.
        if text.startswith("=") and text.endswith("="):
            continue

        sentences = text_to_clean_sentences(text)
        all_sentences.extend(sentences)

    return all_sentences


def main():
    output_dir = Path("data/wikitext_2")
    output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="train",
    )

    val_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="validation",
    )

    test_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )

    train_sentences = convert_wikitext_split_to_sentences(
        train_dataset,
        split_name="train",
    )

    val_sentences = convert_wikitext_split_to_sentences(
        val_dataset,
        split_name="validation",
    )

    test_sentences = convert_wikitext_split_to_sentences(
        test_dataset,
        split_name="test",
    )

    save_lines(
        train_sentences,
        output_dir / "wikitext2_train.txt",
    )

    save_lines(
        val_sentences,
        output_dir / "wikitext2_val.txt",
    )

    save_lines(
        test_sentences,
        output_dir / "wikitext2_test.txt",
    )

    print("Saved WikiText-2 splits:")
    print("Train sentences:", len(train_sentences))
    print("Val sentences:  ", len(val_sentences))
    print("Test sentences: ", len(test_sentences))
    print()
    print("Saved to:")
    print(output_dir / "wikitext2_train.txt")
    print(output_dir / "wikitext2_val.txt")
    print(output_dir / "wikitext2_test.txt")


if __name__ == "__main__":
    main()

Processing WikiText-2 train:   0%|          | 0/36718 [00:00<?, ?it/s]

Processing WikiText-2 validation:   0%|          | 0/3760 [00:00<?, ?it/s]

Processing WikiText-2 test:   0%|          | 0/4358 [00:00<?, ?it/s]

Saved WikiText-2 splits:
Train sentences: 85757
Val sentences:   9046
Test sentences:  10434

Saved to:
data/wikitext_2/wikitext2_train.txt
data/wikitext_2/wikitext2_val.txt
data/wikitext_2/wikitext2_test.txt


## Here i am preparing Tiny Stories tokenizer for the Transformer model. I need to be careful so that the same train, test and validation content is used for both models, in other words, same stories/sentences in both splits. 